# Baseline — PaddleOCR gốc

Notebook này đánh giá mô hình PaddleOCR gốc, chưa fine-tune, trên tập `rec_test.txt`. Kết quả là mốc đối chiếu cho các notebook fine-tune.

**Chuẩn bị trên Kaggle:** gắn Input `baonguyenngoc0902/hcmus-nlp-vietnamese-recognition-resources`, bật GPU và Internet. Trước khi chạy, hãy commit và push repository GitHub của nhóm; Kaggle sẽ clone đúng code/config ở repository đó.


## 1. Cài đặt nền tảng GPU tối thiểu

Cell dưới đây cài PaddlePaddle GPU và các thư viện hỗ trợ. PaddleOCR chưa được cài từ PyPI ở đây, vì notebook sẽ clone đúng mã nguồn PaddleOCR ở bước sau và cài trực tiếp từ source đã được cố định phiên bản.


In [ ]:
import subprocess
import sys

PINNED_PADDLE = "paddlepaddle-gpu==3.0.0"

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
        PINNED_PADDLE,
        "python-Levenshtein>=0.25",
        "PyYAML>=6.0",
        "tqdm>=4.0",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "show", "paddlepaddle-gpu"], check=True)


## 2. Clone mã nguồn PaddleOCR và repository của nhóm

Cell này clone hai repository vào `/kaggle/working`: mã nguồn PaddleOCR chính thức tại tag `v3.0.3`, và repository của nhóm tại nhánh `main`. PaddleOCR được cài ở chế độ editable từ source clone; nhờ đó toàn bộ notebook dùng đúng phiên bản source đã ghi nhận. Repository của nhóm cung cấp config và các hàm dùng chung trong `src/`.

Nếu clone repository nhóm thất bại, kiểm tra repository đã public và mọi thay đổi cần thiết đã được commit, push lên GitHub.


In [ ]:
from pathlib import Path
import subprocess
import sys

WORK_DIR = Path("/kaggle/working")
PADDLEOCR_DIR = WORK_DIR / "PaddleOCR"
PROJECT_DIR = WORK_DIR / "nlp-vietnamese-ocr-recognition"

PADDLEOCR_REPO_URL = "https://github.com/PaddlePaddle/PaddleOCR.git"
PADDLEOCR_REF = "v3.0.3"
PROJECT_REPO_URL = "https://github.com/nguyenngocbao/nlp-vietnamese-ocr-recognition.git"
PROJECT_BRANCH = "main"

def clone_once(url: str, destination: Path, branch: str) -> None:
    if destination.exists():
        print(f"Using existing clone: {destination}")
        return
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", branch, url, str(destination)],
        check=True,
    )

clone_once(PADDLEOCR_REPO_URL, PADDLEOCR_DIR, PADDLEOCR_REF)
clone_once(PROJECT_REPO_URL, PROJECT_DIR, PROJECT_BRANCH)

# Install PaddleOCR from the pinned cloned source, not from a floating wheel.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-e", str(PADDLEOCR_DIR)],
    check=True,
)

def git_commit(directory: Path) -> str:
    return subprocess.check_output(
        ["git", "-C", str(directory), "rev-parse", "HEAD"],
        text=True,
    ).strip()

PADDLEOCR_COMMIT = git_commit(PADDLEOCR_DIR)
PROJECT_COMMIT = git_commit(PROJECT_DIR)

print("PaddleOCR source:", PADDLEOCR_DIR, PADDLEOCR_COMMIT)
print("Project source:", PROJECT_DIR, PROJECT_COMMIT)


## 3. Ghi nhận môi trường chạy và nạp code dùng chung

Cell này cố định seed 42, kiểm tra GPU, in phiên bản phần mềm và thông tin GPU. Nó cũng thêm repository của nhóm vào Python path rồi import `src.metrics` và `src.jsonl`; đây là các hàm dùng chung thay vì sao chép lại logic evaluate trong notebook.


In [ ]:
import os
import platform
import random
import subprocess
import sys
import time

# Set before creating the OCR model. BOS is useful if Hugging Face is unavailable.
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "BOS"

import numpy as np
import paddle
import paddleocr

# Make shared utilities from the cloned group repository importable.
sys.path.insert(0, str(PROJECT_DIR))
from src.jsonl import read_predictions
from src.metrics import recognition_metrics

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
paddle.seed(SEED)

assert paddle.is_compiled_with_cuda(), (
    "PaddlePaddle has no CUDA support. Enable a Kaggle GPU and rerun installation."
)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PaddlePaddle:", paddle.__version__)
print("PaddleOCR:", paddleocr.__version__)
subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    check=True,
)


## 4. Tìm và giải nén dữ liệu

Dữ liệu Kaggle nằm trong thư mục chỉ đọc `/kaggle/input`, nên cell này giải nén `vi_rec_100k.zip` sang `/kaggle/working/data`. Các kết quả nhỏ được ghi vào thư mục `results/` của repository nhóm đã clone, để có thể tải về và commit lại sau khi chạy.


In [ ]:
import zipfile

DATASET_DIR = Path("/kaggle/input/hcmus-nlp-vietnamese-recognition-resources")
WORK_DIR = Path("/kaggle/working")
DATA_DIR = WORK_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_DIR.exists(), "Attach the Kaggle dataset in the Input panel."

print("Dataset files:")
for path in sorted(item for item in DATASET_DIR.rglob("*") if item.is_file()):
    print(" -", path.relative_to(DATASET_DIR))

zip_files = sorted(DATASET_DIR.rglob("vi_rec_100k.zip"))
assert len(zip_files) == 1, f"Expected one vi_rec_100k.zip, found: {zip_files}"

marker = DATA_DIR / ".extracted_from_vi_rec_100k"
if not marker.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_files[0], "r") as archive:
        archive.extractall(DATA_DIR)
    marker.touch()

dict_files = sorted(DATASET_DIR.rglob("vi_dict.txt"))
assert len(dict_files) == 1, f"Expected one vi_dict.txt, found: {dict_files}"
DICT_PATH = dict_files[0]

def find_exactly_one(root: Path, name: str) -> Path:
    matches = sorted(root.rglob(name))
    assert len(matches) == 1, f"Expected one {name}, found: {matches}"
    return matches[0]

TRAIN_LABELS = find_exactly_one(DATA_DIR, "rec_train.txt")
VAL_LABELS = find_exactly_one(DATA_DIR, "rec_val.txt")
TEST_LABELS = find_exactly_one(DATA_DIR, "rec_test.txt")

print("train labels:", TRAIN_LABELS)
print("val labels:", VAL_LABELS)
print("test labels:", TEST_LABELS)
print("dictionary characters:", len(DICT_PATH.read_text(encoding="utf-8").splitlines()))


## 5. Kiểm tra các file nhãn

Mỗi dòng nhãn có dạng `đường_dẫn_ảnh<TAB>văn_bản_đúng`. Cell dưới đây kiểm tra số lượng mẫu ở ba split theo đề bài: 93.997 train, 3.000 validation và 3.003 test. Nếu kiểm tra này thất bại, cần sửa dữ liệu/đường dẫn trước khi đánh giá model.


In [ ]:
def read_label_lines(path: Path) -> list[str]:
    lines = path.read_text(encoding="utf-8").splitlines()
    assert all("\t" in line for line in lines), f"Invalid label format in {path}"
    return lines

split_lines = {
    "train": read_label_lines(TRAIN_LABELS),
    "val": read_label_lines(VAL_LABELS),
    "test": read_label_lines(TEST_LABELS),
}
for split, lines in split_lines.items():
    print(f"{split}: {len(lines):,} labels")

assert len(split_lines["train"]) == 93_997
assert len(split_lines["val"]) == 3_000
assert len(split_lines["test"]) == 3_003
print("First three test labels:")
print("\n".join(split_lines["test"][:3]))


## 6. Ghép nhãn với ảnh test

File nhãn chứa đường dẫn ảnh tương đối. Cell này chuyển chúng thành đường dẫn ảnh thật sau khi giải nén, rồi xác nhận mọi ảnh của tập test đều tồn tại. Chỉ `rec_test.txt` được dùng để tính metric cuối cùng của baseline.


In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

image_by_name: dict[str, Path] = {}
for image_path in DATA_DIR.rglob("*"):
    if image_path.is_file() and image_path.suffix.lower() in IMAGE_SUFFIXES:
        image_by_name.setdefault(image_path.name, image_path)

def resolve_image_path(raw_path: str) -> Path:
    relative_path = Path(raw_path.replace("\\", "/").lstrip("./"))
    for candidate in [DATA_DIR / relative_path, TEST_LABELS.parent / relative_path]:
        if candidate.is_file():
            return candidate
    if relative_path.name in image_by_name:
        return image_by_name[relative_path.name]
    raise FileNotFoundError(f"Cannot resolve image path: {raw_path}")

test_examples = []
for line in split_lines["test"]:
    raw_path, gt = line.split("\t", 1)
    test_examples.append({"image": str(resolve_image_path(raw_path)), "gt": gt})

assert len(test_examples) == 3_003
assert all(Path(row["image"]).is_file() for row in test_examples)
print("Resolved test images:", len(test_examples))
print("Example:", test_examples[0])


## 7. Đọc config baseline từ repository và tải model gốc

Cell này đọc file config baseline từ repository nhóm để lấy tên model, sau đó tải model chính thức `latin_PP-OCRv5_mobile_rec` chưa fine-tune. Vì tiếng Việt sử dụng hệ ký tự Latin, đây là model baseline phù hợp. Việc đọc config trực tiếp bảo đảm notebook và source repository dùng cùng một thiết lập.


In [ ]:
import json
import yaml
from paddleocr import TextRecognition

BASELINE_CONFIG_PATH = (
    PROJECT_DIR / "configs/baseline/paddleocr_original_ppocrv5_mobile.yml"
)
with BASELINE_CONFIG_PATH.open(encoding="utf-8") as file:
    baseline_config = yaml.safe_load(file)

MODEL_NAME = baseline_config["model"]
BATCH_SIZE = 64

print("Baseline config:", BASELINE_CONFIG_PATH)
print("Model:", MODEL_NAME)

model_init_start = time.perf_counter()
recognizer = TextRecognition(
    model_name=MODEL_NAME,
    device="gpu:0",
    engine="paddle_static",
)
model_init_seconds = time.perf_counter() - model_init_start

sample_result = recognizer.predict(
    input=[test_examples[0]["image"]],
    batch_size=1,
)[0]
sample_payload = sample_result.json
if isinstance(sample_payload, str):
    sample_payload = json.loads(sample_payload)

print(sample_payload)
assert "res" in sample_payload and "rec_text" in sample_payload["res"]


## 8. Dự đoán tập test và xuất prediction JSONL

Ảnh được dự đoán theo batch để tận dụng GPU. Mỗi dòng JSONL có ba trường bắt buộc `image`, `gt` và `pred`. File được ghi vào `results/` của repository nhóm clone trên Kaggle; notebook cũng dùng hàm `read_predictions` từ `src/` để xác nhận file JSONL hợp lệ.


In [ ]:
from tqdm.auto import tqdm

def text_from_result(result) -> str:
    payload = result.json
    if isinstance(payload, str):
        payload = json.loads(payload)
    return str(payload["res"]["rec_text"])

prediction_start = time.perf_counter()
prediction_rows = []

for start in tqdm(range(0, len(test_examples), BATCH_SIZE), desc="Predicting test set"):
    batch = test_examples[start : start + BATCH_SIZE]
    results = recognizer.predict(
        input=[row["image"] for row in batch],
        batch_size=len(batch),
    )
    assert len(results) == len(batch), "Prediction count differs from input count."

    for example, result in zip(batch, results):
        prediction_rows.append(
            {"image": example["image"], "gt": example["gt"], "pred": text_from_result(result)}
        )

prediction_seconds = time.perf_counter() - prediction_start
assert len(prediction_rows) == len(test_examples)

PREDICTIONS_PATH = PROJECT_DIR / baseline_config["output"]["predictions"]
PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
with PREDICTIONS_PATH.open("w", encoding="utf-8") as file:
    for row in prediction_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

assert sum(1 for _ in read_predictions(PREDICTIONS_PATH)) == len(prediction_rows)

print(f"Prediction time: {prediction_seconds:.2f} seconds")
print("Predictions:", PREDICTIONS_PATH)


## 9. Tính metric bằng utility dùng chung

Cell này gọi `recognition_metrics` từ `src.metrics` của repository nhóm để tính `acc` và `norm_edit_dis` trên toàn bộ `rec_test.txt`. File metrics cũng lưu commit PaddleOCR và commit repository nhóm, giúp xác định chính xác code nào đã tạo ra kết quả.


In [ ]:
metrics = {
    "model": MODEL_NAME,
    "split": "rec_test.txt",
    **recognition_metrics(prediction_rows),
    "model_init_seconds": model_init_seconds,
    "prediction_seconds": prediction_seconds,
    "seed": SEED,
    "paddleocr_version": paddleocr.__version__,
    "paddlepaddle_version": paddle.__version__,
    "paddleocr_commit": PADDLEOCR_COMMIT,
    "project_commit": PROJECT_COMMIT,
}

METRICS_PATH = RESULTS_DIR / "metrics_baseline_latin_ppocrv5_mobile.json"
METRICS_PATH.write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("Metrics:", METRICS_PATH)


## 10. Kiểm tra lỗi và lấy output

Cell cuối in tối đa 20 mẫu dự đoán sai đầu tiên để kiểm tra nhanh. Sau khi chạy xong, tải file JSONL và metrics JSON từ Kaggle Output; sau đó copy chúng về repository local, kiểm tra rồi commit nếu dung lượng phù hợp. Không push trực tiếp từ Kaggle.


In [ ]:
incorrect_rows = [row for row in prediction_rows if row["gt"] != row["pred"]]

print(f"Incorrect predictions: {len(incorrect_rows):,} / {len(prediction_rows):,}")
for row in incorrect_rows[:20]:
    print(json.dumps(row, ensure_ascii=False))

print("Files to download:")
for path in [PREDICTIONS_PATH, METRICS_PATH]:
    print(f"{path} ({path.stat().st_size / 1024:.1f} KB)")
